# Bangladesh: 0221_climate_bootcamp_session6

Static adaptation only; Earth Engine/data/asset execution not verified.

Original scientific assumptions and date windows retained; use the national workflow for QA improvements.

Review scale, missing data, units, and any training/calibration requirements before interpreting results.

May contain tutorial-specific assets or imported training features; inspect dependencies before execution.

In [ ]:
# Bangladesh adaptation: 0221_climate_bootcamp_session6
# Original author: Amirhossein Ahrari; see original attribution below and originals/.
from bdgee.core import initialize, boundaries
from bdgee.notebook import export_outputs, upload_local
initialize()  # Run `earthengine authenticate` first; project is configured in bdgee.
BD_COUNTRY, BD_DISTRICTS = boundaries()
BD_ROI = BD_COUNTRY.geometry()
BD_BBOX = [88.0, 20.5, 92.7, 26.7]  # search only, not analysis boundary
roi = geometry = border = iran = tehran = province = loc = point = BD_ROI
# Run from the repository root; all local outputs are isolated by source.
from pathlib import Path
import os
_REPO_ROOT = Path.cwd()
_OUTPUT_DIR = _REPO_ROOT / 'outputs' / 'climate' / '0221_climate_bootcamp_session6'
_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(_OUTPUT_DIR)


In [ ]:
"""06_class6_xclim_indices.ipynb

Tutorial Code by Amirhossein Ahrari
YouTube: https://www.youtube.com/@amirhosseinahrarigee
Tutorial Video: Climate Data Analysis Bootcamp using GEE and Python - Session6: Climate Extremes with XClim
This code is part of a tutorial series on Earth Engine programming techniques
presented on the Amirhossein Ahrari YouTube channel. You are free to use and modify
this code for academic and non-academic purposes. Don't forget to subscribe to
the Amirhossein Ahrari channel and follow the videos to support the instructor

"""
import ee
None
None
'# **Study Area**'
import geemap
map = geemap.Map()
map
point = BD_ROI
point
roi = BD_ROI
roi
map.addLayer(roi, {}, 'Iran')
'# **ERA-5 Image Collection**'
collection = ee.ImageCollection('ECMWF/ERA5/DAILY').filterDate('2010', '2016').select(['minimum_2m_air_temperature', 'maximum_2m_air_temperature', 'total_precipitation'], ['min_temp', 'max_temp', 'pr'])
collection
'# **to Xarray**'
from xee import helpers
from shapely.geometry import shape
import xarray as xr
grid_params = helpers.fit_geometry(geometry=shape(roi.getInfo()), grid_crs='EPSG:4326', grid_scale=(0.3, -0.3))
ds = xr.open_dataset(collection, engine='ee', **grid_params)
ds_temp = ds[['min_temp', 'max_temp']] - 273.15
ds_pr = ds['pr'] * 1000
'# **Clip**'
import geopandas as gpd
roi_gdf = geemap.ee_to_gdf(ee.FeatureCollection(roi))
import rioxarray
temp_clip = ds_temp.rio.clip(roi_gdf.geometry, roi_gdf.crs)
pr_clip = ds_pr.rio.clip(roi_gdf.geometry, roi_gdf.crs)
'# **Temperature**'
temp_mean = temp_clip.mean(dim='time')
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 5))
temp_mean.min_temp.plot.contourf(x='x', y='y', robust=True, ax=ax[0], levels=10, cmap='Blues')
temp_mean.max_temp.plot.contourf(x='x', y='y', robust=True, ax=ax[1], levels=10, cmap='Reds')
plt.tight_layout()
import xclim.indicators.atmos as xin
temp_clip.min_temp.attrs['units'] = 'degC'
temp_clip.max_temp.attrs['units'] = 'degC'
temp_clip
hw_freq = xin.heat_wave_frequency(tasmin=temp_clip.min_temp, tasmax=temp_clip.max_temp, thresh_tasmin='30.0 degC', thresh_tasmax='40.0 degC')
hw_freq.plot.contourf(x='x', y='y', robust=True, cmap='turbo', levels=8, col='time', col_wrap=3)
hw_index = xin.heat_wave_index(tasmax=temp_clip.max_temp, thresh='40 degC', window=7)
hw_index.plot.contourf(x='x', y='y', robust=True, cmap='turbo', levels=8, col='time', col_wrap=3)
'# **Precipitation**'
pr_mean = pr_clip.mean(dim='time')
pr_mean.plot.contourf(x='x', y='y', robust=True, levels=8)
pr_clip.attrs['units'] = 'mm/d'
dry_spell = xin.dry_spell_frequency(pr=pr_clip, thresh='0.5 mm', window=7)
dry_spell.plot.contourf(x='x', y='y', col='time', col_wrap=3, levels=8, cmap='turbo', robust=True)
dry_days = xin.dry_days(pr=pr_clip, thresh='0.2 mm/d')
dry_days.plot.contourf(x='x', y='y', robust=True, cmap='turbo', levels=8, col='time', col_wrap=3)


In [ ]:

# Review these task plans; task creation does not start exports.
os.chdir(_REPO_ROOT)
bd_tasks = export_outputs(globals(), 'climate', '0221_climate_bootcamp_session6', scale=1000)
# Set an appropriate native scale above before creating plans.
# for task in bd_tasks: task.start()
# upload_local(_OUTPUT_DIR, 'climate')  # explicit upload; requires Drive OAuth scope
